<a href="https://colab.research.google.com/github/mindioanni/LandLevelTools/blob/main/ANTEX_Comparisons_V1_20260219.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GNSS ANTEX (PCO/PCV) Analysis & Visualization (Colab)
_______________________________________________________________________________

## A) Credits

### 1. Conceptualization / Author / Developer (User)
**Ioannis Mintourakis,**

Postdoctoral Researcher (Sea Level / Geodesy)

Hellenic National Tsunami Warning Centre (HL-NTWC)  
Institute of Geodynamics, National Observatory of Athens, Greece  
cellphone : ++30-6942556965  
e-mail : i.mintourakis@noa.gr / mindioanni@gmail.com

Greetings from: 37° 58' 30.28"N  &nbsp;&nbsp;&nbsp;&nbsp;23° 46' 47.90"E

### 2. Author / Developer (Assistant)
**R**

The name **"R"** is inspired by the robot characters in the **Isaac Asimov** series.

The real identity of my Assistant is:

OpenAI. (2026, February 19). ChatGPT (**GPT-5.2 Thinking**) [Large language model]. Retrieved February 19, 2026, from https://chatgpt.com/

_______________________________________________________________________________

## B) Purpose
This notebook performs **analysis, visualization, and QC-style documentation** of **GNSS ANTEX (`.atx`) antenna calibrations**, focusing on:
- **PCO offsets** (North/East/Up) per frequency,
- **PCV** as a function of **zenith** (NOAZI / 1D),
- **PCV azimuth × zenith grids** (2D) when available,
- optional **differences between two ANTEX calibrations** (ΔPCO, ΔPCV) for **common frequencies**.

It produces a **single HTML report** (with plots saved under `assets/`) to document:
- parsed antennas and frequency availability,
- extracted calibration grids (DAZI, zenith grid),
- plots and summary tables.

_______________________________________________________________________________

## C) Quick start
1) Run cells sequentially: **CELL 1 → CELL 11**.

2) Inputs:
   - Upload one or more `.atx` files (or mount Google Drive and provide paths).

3) Frequency selection:
   - **a)** common frequencies (intersection), or  
   - **b)** all available frequencies (union), or  
   - **c)** manual list (comma-separated).

4) Mode selection:
   - **NOAZI (1D)** curves vs zenith, or  
   - **2D** azimuth × zenith heatmaps (where DAZI>0 and azimuth bins exist).

5) Output:
   - Report folder: **`/content/report/`** containing `report.html` and `assets/`
   - Downloadable zip: **`/content/report.zip`**

In [1]:
# =========================
# CELL 1 — Setup: imports
# =========================

import os
import re
import io
import json
import math
import base64
from pathlib import Path
from datetime import datetime, timezone
from html import escape as html_escape

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Imports OK")
print("Python:", f"{os.sys.version_info.major}.{os.sys.version_info.minor}.{os.sys.version_info.micro}")
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", plt.matplotlib.__version__)


Imports OK
Python: 3.12.12
numpy: 2.0.2
pandas: 2.2.2
matplotlib: 3.10.0


In [2]:
# =========================================
# CELL 2 — Input: upload OR Drive mount
# =========================================

from google.colab import files, drive

WORKDIR = Path("/content/atx_work")
WORKDIR.mkdir(parents=True, exist_ok=True)

def list_atx_files(folder: Path):
    return sorted([p for p in folder.glob("*.atx") if p.is_file()])

print("Choose input method:")
print("  1) Upload .atx files")
print("  2) Mount Google Drive and copy .atx files from a folder")

method = input("Enter 1 or 2: ").strip()

if method == "1":
    uploaded = files.upload()  # user selects one or more .atx
    # Save uploaded files into WORKDIR (Colab already writes them to /content)
    for name in uploaded.keys():
        src = Path("/content") / name
        dst = WORKDIR / name
        if src.resolve() != dst.resolve():
            dst.write_bytes(src.read_bytes())
    print(f"Uploaded files copied to: {WORKDIR}")

elif method == "2":
    drive.mount("/content/drive")
    src_folder = input("Enter Drive folder path containing .atx files (e.g. /content/drive/MyDrive/ANTEX): ").strip()
    src_folder = Path(src_folder)

    if not src_folder.exists():
        raise FileNotFoundError(f"Folder does not exist: {src_folder}")

    atx_src = list_atx_files(src_folder)
    if len(atx_src) == 0:
        raise FileNotFoundError(f"No .atx files found in: {src_folder}")

    for p in atx_src:
        (WORKDIR / p.name).write_bytes(p.read_bytes())
    print(f"Copied {len(atx_src)} .atx files to: {WORKDIR}")

else:
    raise ValueError("Invalid selection. Enter '1' or '2'.")

atx_files = list_atx_files(WORKDIR)
print("\nDetected .atx files:")
for p in atx_files:
    print(" -", p.name)

if len(atx_files) == 0:
    raise RuntimeError("No .atx files available in working directory.")


Choose input method:
  1) Upload .atx files
  2) Mount Google Drive and copy .atx files from a folder
Enter 1 or 2: 1


Saving leigs18_________none.atx to leigs18_________none.atx
Saving LEIGS18_NONE.atx to LEIGS18_NONE.atx
Uploaded files copied to: /content/atx_work

Detected .atx files:
 - LEIGS18_NONE.atx
 - leigs18_________none.atx


In [3]:
# =========================================
# CELL 3 — HTML report engine skeleton
# =========================================

from google.colab import files as colab_files

REPORT_DIR = Path("/content/report")
ASSETS_DIR = REPORT_DIR / "assets"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ASSETS_DIR.mkdir(parents=True, exist_ok=True)

TS_UTC = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
REPORT_HTML = REPORT_DIR / "report.html"

def html_warn(msg: str) -> str:
    return f"<b style='color:#b00020'>WARNING:</b> <b style='color:#b00020'>{html_escape(msg)}</b>"

def report_init(title: str = "ANTEX (PCO/PCV) Report"):
    html = []
    html.append("<!doctype html><html><head><meta charset='utf-8'>")
    html.append(f"<title>{html_escape(title)}</title>")
    html.append("<style>"
                "body{font-family:Arial,Helvetica,sans-serif;max-width:1200px;margin:20px;}"
                "code{background:#f4f4f4;padding:2px 4px;}"
                "table{border-collapse:collapse;}"
                "td,th{border:1px solid #ccc;padding:6px 8px;vertical-align:top;}"
                "hr{margin:18px 0;}"
                "</style>")
    html.append("</head><body>")
    html.append(f"<h1>{html_escape(title)}</h1>")
    html.append(f"<p><b>Generated (UTC):</b> {html_escape(TS_UTC)}</p>")
    html.append("<hr>")
    REPORT_HTML.write_text("\n".join(html), encoding="utf-8")

def report_append(html_fragment: str):
    with REPORT_HTML.open("a", encoding="utf-8") as f:
        f.write("\n" + html_fragment + "\n")

def report_close():
    report_append("</body></html>")

def r_h2(text: str):
    report_append(f"<h2>{html_escape(text)}</h2>")

def r_p(text: str):
    report_append(f"<p>{html_escape(text)}</p>")

def r_pre(text: str):
    report_append(f"<pre>{html_escape(text)}</pre>")

def r_df(df: pd.DataFrame, caption: str | None = None, index: bool = False):
    if caption:
        report_append(f"<p><b>{html_escape(caption)}</b></p>")
    # escape=True ensures safe HTML
    report_append(df.to_html(escape=True, index=index))

def r_fig(fig, filename: str, caption: str | None = None):
    # Save to assets and embed
    out = ASSETS_DIR / filename
    fig.savefig(out, dpi=150, bbox_inches="tight")
    if caption:
        report_append(f"<p><b>{html_escape(caption)}</b></p>")
    report_append(f"<img src='assets/{html_escape(filename)}' style='max-width:1100px;width:100%;height:auto;'>")

def report_download():
    # Ensure report is closed before download
    if REPORT_HTML.exists():
        colab_files.download(str(REPORT_HTML))
    else:
        raise FileNotFoundError("report.html not found.")

# --- Create initial report content (smoke test) ---
report_init("ANTEX (PCO/PCV) Report")
r_h2("Inputs")
r_p(f"Working directory: {WORKDIR}")
r_p("ATX files detected:")
r_pre("\n".join([p.name for p in atx_files]))

# smoke-test table
df_test = pd.DataFrame({"file": [p.name for p in atx_files]})
r_df(df_test, caption="ATX file list", index=False)

# smoke-test figure
fig = plt.figure()
plt.plot([0, 1, 2], [0, 1, 0])
plt.title("Smoke test plot")
r_fig(fig, "smoke_test.png", caption="Smoke test plot (will be replaced later)")
plt.close(fig)

report_close()

print("Report created:", REPORT_HTML)
print("Assets dir:", ASSETS_DIR)
print("Open report in Colab file browser: /content/report/report.html")


Report created: /content/report/report.html
Assets dir: /content/report/assets
Open report in Colab file browser: /content/report/report.html


In [4]:
# =========================================
# CELL 3.1 — Notebook header (Credits / Purpose / Quick start) + write into report.html
# =========================================

from IPython.display import display, Markdown
import markdown as mdlib

md = r"""
# GNSS ANTEX (PCO/PCV) Analysis & Visualization (Colab)
_______________________________________________________________________________

## A) Credits

### 1. Conceptualization / Author / Developer (User)
**Ioannis Mintourakis,**

Postdoctoral Researcher (Sea Level / Geodesy)

Hellenic National Tsunami Warning Centre (HL-NTWC)
Institute of Geodynamics, National Observatory of Athens, Greece
cellphone : ++30-6942556965
e-mail : i.mintourakis@noa.gr / mindioanni@gmail.com

Greetings from: 37° 58' 30.28"N  &nbsp;&nbsp;&nbsp;&nbsp;23° 46' 47.90"E

### 2. Author / Developer (Assistant)
**R**

The name **"R"** is inspired by the robot characters in the **Isaac Asimov** series.

The real identity of my Assistant is:

OpenAI. (2026, February 19). ChatGPT (**GPT-5.2 Thinking**) [Large language model]. Retrieved February 19, 2026, from https://chatgpt.com/

_______________________________________________________________________________

## B) Purpose
This notebook performs **analysis, visualization, and QC-style documentation** of **GNSS ANTEX (`.atx`) antenna calibrations**, focusing on:
- **PCO offsets** (North/East/Up) per frequency,
- **PCV** as a function of **zenith** (NOAZI / 1D),
- **PCV azimuth × zenith grids** (2D) when available,
- optional **differences between two ANTEX calibrations** (ΔPCO, ΔPCV) for **common frequencies**.

It produces a **single HTML report** (with plots saved under `assets/`) to document:
- parsed antennas and frequency availability,
- extracted calibration grids (DAZI, zenith grid),
- plots and summary tables.

_______________________________________________________________________________

## C) Quick start
1) Run cells sequentially: **CELL 1 → CELL 11**.

2) Inputs:
   - Upload one or more `.atx` files (or mount Google Drive and provide paths).

3) Frequency selection:
   - **a)** common frequencies (intersection), or
   - **b)** all available frequencies (union), or
   - **c)** manual list (comma-separated).

4) Mode selection:
   - **NOAZI (1D)** curves vs zenith, or
   - **2D** azimuth × zenith heatmaps (where DAZI>0 and azimuth bins exist).

5) Output:
   - Report folder: **`/content/report/`** containing `report.html` and `assets/`
   - Downloadable zip: **`/content/report.zip`**
"""

# Notebook rendering
display(Markdown(md))

# ---- Write same content into report.html (rendered markdown) ----
report_append("<hr>")
r_h2("Notebook header")

header_html = mdlib.markdown(md, extensions=["extra", "sane_lists"])
report_append(header_html)



# GNSS ANTEX (PCO/PCV) Analysis & Visualization (Colab)
_______________________________________________________________________________

## A) Credits

### 1. Conceptualization / Author / Developer (User)
**Ioannis Mintourakis,**

Postdoctoral Researcher (Sea Level / Geodesy)

Hellenic National Tsunami Warning Centre (HL-NTWC)  
Institute of Geodynamics, National Observatory of Athens, Greece  
cellphone : ++30-6942556965  
e-mail : i.mintourakis@noa.gr / mindioanni@gmail.com

Greetings from: 37° 58' 30.28"N  &nbsp;&nbsp;&nbsp;&nbsp;23° 46' 47.90"E

### 2. Author / Developer (Assistant)
**R**

The name **"R"** is inspired by the robot characters in the **Isaac Asimov** series.

The real identity of my Assistant is:

OpenAI. (2026, February 19). ChatGPT (**GPT-5.2 Thinking**) [Large language model]. Retrieved February 19, 2026, from https://chatgpt.com/

_______________________________________________________________________________

## B) Purpose
This notebook performs **analysis, visualization, and QC-style documentation** of **GNSS ANTEX (`.atx`) antenna calibrations**, focusing on:
- **PCO offsets** (North/East/Up) per frequency,
- **PCV** as a function of **zenith** (NOAZI / 1D),
- **PCV azimuth × zenith grids** (2D) when available,
- optional **differences between two ANTEX calibrations** (ΔPCO, ΔPCV) for **common frequencies**.

It produces a **single HTML report** (with plots saved under `assets/`) to document:
- parsed antennas and frequency availability,
- extracted calibration grids (DAZI, zenith grid),
- plots and summary tables.

_______________________________________________________________________________

## C) Quick start
1) Run cells sequentially: **CELL 1 → CELL 11**.

2) Inputs:
   - Upload one or more `.atx` files (or mount Google Drive and provide paths).

3) Frequency selection:
   - **a)** common frequencies (intersection), or  
   - **b)** all available frequencies (union), or  
   - **c)** manual list (comma-separated).

4) Mode selection:
   - **NOAZI (1D)** curves vs zenith, or  
   - **2D** azimuth × zenith heatmaps (where DAZI>0 and azimuth bins exist).

5) Output:
   - Report folder: **`/content/report/`** containing `report.html` and `assets/`
   - Downloadable zip: **`/content/report.zip`**


In [5]:
# =========================================
# CELL 4 — ANTEX parser (START OF ANTENNA / START OF FREQUENCY)
# =========================================

FLOAT_RE = re.compile(r"[-+]?\d*\.\d+|[-+]?\d+")

def _extract_floats(s: str):
    return [float(x) for x in FLOAT_RE.findall(s)]

def _nzen(zen1: float, zen2: float, dzen: float) -> int:
    return int(round((zen2 - zen1) / dzen)) + 1

def _read_pcv_values_1d(lines, i, need_count):
    vals = []
    while i < len(lines) and len(vals) < need_count:
        line = lines[i]
        if ("END OF FREQUENCY" in line or "START OF FREQUENCY" in line or
            "END OF ANTENNA" in line or "START OF ANTENNA" in line):
            break
        clean = line.replace("NOAZI", " ")
        vals.extend(_extract_floats(clean))
        i += 1
    return vals[:need_count], i

def parse_antex_file(path: Path):
    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()

    # Skip global header
    i = 0
    while i < len(lines) and "END OF HEADER" not in lines[i]:
        i += 1
    if i < len(lines) and "END OF HEADER" in lines[i]:
        i += 1

    antennas = []
    cur_ant = None
    cur_freq = None

    # antenna-level defaults
    ant_dazi = None
    ant_zen = None   # (zen1, zen2, dzen)
    ant_z = None

    while i < len(lines):
        line = lines[i]

        if "START OF ANTENNA" in line:
            cur_ant = {
                "file": path.name,
                "antenna_type": None,
                "serial": None,
                "meta": {},
                "freq": {}
            }
            cur_freq = None
            ant_dazi, ant_zen, ant_z = None, None, None
            i += 1
            continue

        if "END OF ANTENNA" in line:
            if cur_ant is not None:
                antennas.append(cur_ant)
            cur_ant = None
            cur_freq = None
            i += 1
            continue

        if cur_ant is None:
            i += 1
            continue

        if "TYPE / SERIAL NO" in line:
            ant_type = line[0:20].strip()
            serial = line[20:40].strip()
            cur_ant["antenna_type"] = ant_type if ant_type else None
            cur_ant["serial"] = serial if serial else None
            i += 1
            continue

        # antenna-level DAZI / ZEN (common in many ATX)
        if cur_freq is None and line.strip().endswith("DAZI"):
            vals = _extract_floats(line[:20])
            if len(vals) >= 1:
                ant_dazi = float(vals[0])
            i += 1
            continue

        if cur_freq is None and "ZEN1 / ZEN2 / DZEN" in line:
            vals = _extract_floats(line[:30])
            if len(vals) >= 3:
                zen1, zen2, dzen = float(vals[0]), float(vals[1]), float(vals[2])
                ant_zen = (zen1, zen2, dzen)
                ant_z = np.linspace(zen1, zen2, _nzen(zen1, zen2, dzen))
            i += 1
            continue

        if "START OF FREQUENCY" in line:
            code = line[0:3].strip()
            if not code:
                code = line.strip().split()[0].strip()

            cur_freq = {
                "pco": None,
                "dazi": ant_dazi,
                "zen1": None,
                "zen2": None,
                "dzen": None,
                "z": None,
                "pcv_noazi": None,
                "pcv_azi": {}
            }
            if ant_zen is not None:
                zen1, zen2, dzen = ant_zen
                cur_freq["zen1"], cur_freq["zen2"], cur_freq["dzen"] = zen1, zen2, dzen
                cur_freq["z"] = ant_z.copy() if ant_z is not None else None

            cur_ant["freq"][code] = cur_freq
            i += 1
            continue

        if "END OF FREQUENCY" in line:
            cur_freq = None
            i += 1
            continue

        if cur_freq is None:
            i += 1
            continue

        # frequency-level overrides
        if line.strip().endswith("DAZI"):
            vals = _extract_floats(line[:20])
            if len(vals) >= 1:
                cur_freq["dazi"] = float(vals[0])
            i += 1
            continue

        if "ZEN1 / ZEN2 / DZEN" in line:
            vals = _extract_floats(line[:30])
            if len(vals) >= 3:
                zen1, zen2, dzen = float(vals[0]), float(vals[1]), float(vals[2])
                cur_freq["zen1"], cur_freq["zen2"], cur_freq["dzen"] = zen1, zen2, dzen
                cur_freq["z"] = np.linspace(zen1, zen2, _nzen(zen1, zen2, dzen))
            i += 1
            continue

        if "NORTH / EAST / UP" in line:
            vals = _extract_floats(line[:60])
            if len(vals) >= 3:
                cur_freq["pco"] = (vals[0], vals[1], vals[2])
            i += 1
            continue

        if "NOAZI" in line:
            if cur_freq["z"] is None:
                i += 1
                continue
            need = len(cur_freq["z"])
            vals0 = _extract_floats(line.replace("NOAZI", " "))
            vals, j = _read_pcv_values_1d(lines, i + 1, max(0, need - len(vals0)))
            allv = (vals0 + vals)[:need]
            if len(allv) == need:
                cur_freq["pcv_noazi"] = np.array(allv, dtype=float)
            i = j
            continue

        if cur_freq["z"] is not None and cur_freq["dazi"] is not None and cur_freq["dazi"] > 0:
            floats = _extract_floats(line)
            if len(floats) >= 2:
                azi = float(floats[0])
                vals0 = floats[1:]
                need = len(cur_freq["z"])
                vals, j = _read_pcv_values_1d(lines, i + 1, max(0, need - len(vals0)))
                allv = (vals0 + vals)[:need]
                if len(allv) == need:
                    cur_freq["pcv_azi"][azi] = np.array(allv, dtype=float)
                    i = j
                    continue

        i += 1

    return antennas


In [6]:
# =========================================
# CELL 5 — Parse all .atx and produce summary tables (and write to report)
# =========================================

# Parse
all_antennas = []
for p in atx_files:
    ants = parse_antex_file(p)
    all_antennas.extend(ants)

if len(all_antennas) == 0:
    raise RuntimeError("No antennas parsed from the provided .atx files.")

# Build a stable antenna key
def antenna_key(a):
    t = a.get("antenna_type") or "UNKNOWN"
    s = a.get("serial") or "NONE"
    return f"{t}__{s}__{a.get('file','UNKNOWN')}"

# Convert to dict keyed by antenna_key (keeps all)
ant_data = {antenna_key(a): a for a in all_antennas}

# Summary: antennas
df_ant = pd.DataFrame([
    {
        "antenna_key": k,
        "file": v["file"],
        "antenna_type": v["antenna_type"],
        "serial": v["serial"],
        "n_freq": len(v["freq"])
    }
    for k, v in ant_data.items()
]).sort_values(["file", "antenna_type", "serial"], na_position="last")

# Summary: frequencies
rows = []
for k, a in ant_data.items():
    for fcode, f in a["freq"].items():
        zinfo = None
        if f["zen1"] is not None:
            zinfo = f'{f["zen1"]}-{f["zen2"]} step {f["dzen"]}'
        rows.append({
            "antenna_key": k,
            "freq": fcode,
            "PCO_present": f["pco"] is not None,
            "DAZI": f["dazi"],
            "zenith_grid": zinfo,
            "NOAZI_present": f["pcv_noazi"] is not None,
            "n_azi_bins": len(f["pcv_azi"]) if f["pcv_azi"] is not None else 0
        })

df_freq = pd.DataFrame(rows).sort_values(["antenna_key", "freq"])

# Write to HTML report (new sections appended)
report_append("<hr>")
r_h2("Parser summary")
r_p(f"Parsed antennas: {len(df_ant)}")
r_df(df_ant, caption="Antennas found", index=False)
r_df(df_freq, caption="Frequencies summary", index=False)

print("Parsed antennas:", len(df_ant))
display(df_ant)
print("\nFrequencies summary:")
display(df_freq)


Parsed antennas: 2


,antenna_key,file,antenna_type,serial,n_freq
0,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,LEIGS18_NONE.atx,LEIGS18 NONE,None,4
1,LEIGS18 NONE__NONE__leigs18_________no...,leigs18_________none.atx,LEIGS18 NONE,None,10



Frequencies summary:


,antenna_key,freq,PCO_present,DAZI,zenith_grid,NOAZI_present,n_azi_bins
0,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G01,True,5.0,0.0-90.0 step 5.0,True,73
1,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G02,True,5.0,0.0-90.0 step 5.0,True,73
2,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R01,True,5.0,0.0-90.0 step 5.0,True,73
3,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R02,True,5.0,0.0-90.0 step 5.0,True,73
6,LEIGS18 NONE__NONE__leigs18_________no...,C02,True,5.0,0.0-90.0 step 5.0,True,73
8,LEIGS18 NONE__NONE__leigs18_________no...,C06,True,5.0,0.0-90.0 step 5.0,True,73
7,LEIGS18 NONE__NONE__leigs18_________no...,E06,True,5.0,0.0-90.0 step 5.0,True,73
11,LEIGS18 NONE__NONE__leigs18_________no...,E07,True,5.0,0.0-90.0 step 5.0,True,73
12,LEIGS18 NONE__NONE__leigs18_________no...,E08,True,5.0,0.0-90.0 step 5.0,True,73
4,LEIGS18 NONE__NONE__leigs18_________no...,G01,True,5.0,0.0-90.0 step 5.0,True,73


In [7]:
# =========================================
# CELL 6 — User selection: frequency codes + mode (NOAZI vs 2D)
# =========================================

import numpy as np

# --- helper: list freqs per antenna_key ---
freqs_by_ant = {k: sorted(list(v["freq"].keys())) for k, v in ant_data.items()}

all_freqs = sorted(set().union(*[set(v) for v in freqs_by_ant.values()])) if freqs_by_ant else []
common_freqs = sorted(set.intersection(*[set(v) for v in freqs_by_ant.values()])) if len(freqs_by_ant) >= 2 else all_freqs

print("Antennas parsed:")
for idx, k in enumerate(ant_data.keys()):
    print(f"  [{idx}] {k}")

print("\nFrequencies summary:")
print("  Common frequencies:", ", ".join(common_freqs) if common_freqs else "(none)")
print("  All frequencies:", ", ".join(all_freqs) if all_freqs else "(none)")

# --- selection of frequencies ---
print("\nSelect frequency set:")
print("  a) all common")
print("  b) all available")
print("  c) manual list (comma-separated, e.g. G01,G02,E06)")
freq_choice = input("Enter a/b/c: ").strip().lower()

if freq_choice == "a":
    selected_freqs = common_freqs
elif freq_choice == "b":
    selected_freqs = all_freqs
elif freq_choice == "c":
    manual = input("Enter comma-separated frequency codes: ").strip()
    selected_freqs = [x.strip().upper() for x in manual.split(",") if x.strip()]
    # keep only those that exist in any antenna
    selected_freqs = [f for f in selected_freqs if f in all_freqs]
else:
    raise ValueError("Invalid selection for frequencies. Use a/b/c.")

if len(selected_freqs) == 0:
    raise RuntimeError("No frequencies selected (empty after filtering).")

# --- mode selection ---
print("\nSelect mode:")
print("  1) NOAZI (1D)")
print("  2) 2D azimuth grid (only where DAZI>0 and azimuth bins exist)")
mode_choice = input("Enter 1 or 2: ").strip()

if mode_choice == "1":
    selected_mode = "NOAZI"
elif mode_choice == "2":
    selected_mode = "2D"
else:
    raise ValueError("Invalid mode. Use 1 or 2.")

# --- save selections (global variables used by next cells) ---
SELECTIONS = {
    "selected_freqs": selected_freqs,
    "selected_mode": selected_mode,
    "all_freqs": all_freqs,
    "common_freqs": common_freqs
}

# --- write to report ---
report_append("<hr>")
r_h2("User selections")
r_p(f"Frequency selection option: {freq_choice}")
r_p(f"Selected frequencies ({len(selected_freqs)}): {', '.join(selected_freqs)}")
r_p(f"Selected mode: {selected_mode}")

# also show in notebook
print("\nFinal selections:")
print("  selected_freqs:", selected_freqs)
print("  selected_mode:", selected_mode)

display(pd.DataFrame({"selected_freqs": selected_freqs}))


Antennas parsed:
  [0] LEIGS18         NONE__NONE__LEIGS18_NONE.atx
  [1] LEIGS18         NONE__NONE__leigs18_________none.atx

Frequencies summary:
  Common frequencies: G01, G02, R01, R02
  All frequencies: C02, C06, E06, E07, E08, G01, G02, G05, R01, R02

Select frequency set:
  a) all common
  b) all available
  c) manual list (comma-separated, e.g. G01,G02,E06)
Enter a/b/c: b

Select mode:
  1) NOAZI (1D)
  2) 2D azimuth grid (only where DAZI>0 and azimuth bins exist)
Enter 1 or 2: 2

Final selections:
  selected_freqs: ['C02', 'C06', 'E06', 'E07', 'E08', 'G01', 'G02', 'G05', 'R01', 'R02']
  selected_mode: 2D


,selected_freqs
0,C02
1,C06
2,E06
3,E07
4,E08
5,G01
6,G02
7,G05
8,R01
9,R02


In [8]:
# =========================================
# CELL 7 — Plot: PCO bar chart (N/E/U) per antenna, per selected frequencies
# =========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

selected_freqs = SELECTIONS["selected_freqs"]

def build_pco_df(ant_key: str, ant_dict: dict, freqs: list[str]) -> pd.DataFrame:
    rows = []
    for f in freqs:
        fdata = ant_dict["freq"].get(f)
        if fdata is None:
            rows.append({"freq": f, "N_mm": np.nan, "E_mm": np.nan, "U_mm": np.nan, "present": False})
            continue
        pco = fdata.get("pco")
        if pco is None:
            rows.append({"freq": f, "N_mm": np.nan, "E_mm": np.nan, "U_mm": np.nan, "present": False})
        else:
            # PCO in ANTEX is typically in mm; keep as-is (mm)
            rows.append({"freq": f, "N_mm": float(pco[0]), "E_mm": float(pco[1]), "U_mm": float(pco[2]), "present": True})
    df = pd.DataFrame(rows)
    df.insert(0, "antenna_key", ant_key)
    return df

def plot_pco_bars(df: pd.DataFrame, ant_key: str):
    # Only plot rows where PCO exists
    dfp = df[df["present"]].copy()
    if dfp.empty:
        print(f"No PCO values to plot for {ant_key}")
        return None, None

    x = np.arange(len(dfp))
    w = 0.25

    fig = plt.figure(figsize=(12, 4.5))
    plt.bar(x - w, dfp["N_mm"].values, width=w, label="North (mm)")
    plt.bar(x,     dfp["E_mm"].values, width=w, label="East (mm)")
    plt.bar(x + w, dfp["U_mm"].values, width=w, label="Up (mm)")
    plt.xticks(x, dfp["freq"].values)
    plt.xlabel("Frequency code")
    plt.ylabel("PCO (mm)")
    plt.title(f"PCO offsets (N/E/U) — {ant_key}")
    plt.legend()
    plt.grid(True, axis="y", linestyle="--", linewidth=0.5)
    plt.tight_layout()

    # filename-safe
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", ant_key)[:120]
    fname = f"pco_{safe}.png"
    return fig, fname

# --- generate per-antenna tables + plots, and write to report ---
report_append("<hr>")
r_h2("PCO offsets (N/E/U)")

all_pco_tables = []

for ant_key, ant in ant_data.items():
    df_pco = build_pco_df(ant_key, ant, selected_freqs)
    all_pco_tables.append(df_pco)

    r_h2(f"PCO — {ant_key}")
    r_df(df_pco.drop(columns=["antenna_key"]), caption="PCO table (mm); missing where frequency not present", index=False)

    fig, fname = plot_pco_bars(df_pco, ant_key)
    if fig is not None:
        r_fig(fig, fname, caption="PCO offsets per frequency (mm)")
        plt.close(fig)

# notebook outputs
df_pco_all = pd.concat(all_pco_tables, ignore_index=True)
print("PCO tables created for", df_pco_all["antenna_key"].nunique(), "antennas.")
display(df_pco_all)


PCO tables created for 2 antennas.


,antenna_key,freq,N_mm,E_mm,U_mm,present
0,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,C02,NaN,NaN,NaN,False
1,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,C06,NaN,NaN,NaN,False
2,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,E06,NaN,NaN,NaN,False
3,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,E07,NaN,NaN,NaN,False
4,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,E08,NaN,NaN,NaN,False
5,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G01,-1.00,-0.28,99.91,True
6,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G02,-0.49,2.44,107.44,True
7,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G05,NaN,NaN,NaN,False
8,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R01,-1.00,-0.28,99.91,True
9,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R02,-0.49,2.44,107.44,True


In [9]:
# =========================================
# CELL 8 — Plot: PCV vs zenith (NOAZI) per antenna for selected frequencies
# =========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

selected_freqs = SELECTIONS["selected_freqs"]

def build_pcv_noazi_df(ant_key: str, ant_dict: dict, freqs: list[str]) -> pd.DataFrame:
    """
    One row per frequency with metadata, plus stores arrays in dict for plotting.
    Returns df (tabular summary) and a dict freq->(z, pcv_noazi).
    """
    rows = []
    series = {}
    for f in freqs:
        fdata = ant_dict["freq"].get(f)
        if fdata is None:
            rows.append({
                "freq": f, "present": False, "DAZI": np.nan, "zen1": np.nan, "zen2": np.nan, "dzen": np.nan,
                "n_z": 0, "NOAZI_present": False
            })
            continue

        z = fdata.get("z")
        pcv = fdata.get("pcv_noazi")
        dazi = fdata.get("dazi")

        ok = (z is not None) and (pcv is not None) and (len(z) == len(pcv))
        rows.append({
            "freq": f,
            "present": True,
            "DAZI": dazi if dazi is not None else np.nan,
            "zen1": fdata.get("zen1") if fdata.get("zen1") is not None else np.nan,
            "zen2": fdata.get("zen2") if fdata.get("zen2") is not None else np.nan,
            "dzen": fdata.get("dzen") if fdata.get("dzen") is not None else np.nan,
            "n_z": int(len(z)) if z is not None else 0,
            "NOAZI_present": bool(ok)
        })
        if ok:
            series[f] = (np.array(z, dtype=float), np.array(pcv, dtype=float))

    df = pd.DataFrame(rows)
    df.insert(0, "antenna_key", ant_key)
    return df, series

def plot_pcv_noazi(series: dict, ant_key: str):
    if len(series) == 0:
        print(f"No NOAZI PCV series to plot for {ant_key}")
        return None, None

    fig = plt.figure(figsize=(12, 5))
    for f, (z, pcv) in series.items():
        plt.plot(z, pcv, label=f)

    plt.xlabel("Zenith angle (deg)")
    plt.ylabel("PCV (mm)")
    plt.title(f"PCV (NOAZI) vs Zenith — {ant_key}")
    plt.grid(True, linestyle="--", linewidth=0.5)
    plt.legend(ncol=3, fontsize=9)
    plt.tight_layout()

    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", ant_key)[:120]
    fname = f"pcv_noazi_{safe}.png"
    return fig, fname

# --- generate per-antenna summary table + plot, and write to report ---
report_append("<hr>")
r_h2("PCV (NOAZI) vs Zenith")

all_noazi_tables = []

for ant_key, ant in ant_data.items():
    df_noazi, series = build_pcv_noazi_df(ant_key, ant, selected_freqs)
    all_noazi_tables.append(df_noazi)

    r_h2(f"NOAZI PCV — {ant_key}")
    r_df(df_noazi.drop(columns=["antenna_key"]), caption="NOAZI PCV availability and zenith grid metadata", index=False)

    fig, fname = plot_pcv_noazi(series, ant_key)
    if fig is not None:
        r_fig(fig, fname, caption="PCV (NOAZI) vs zenith for selected frequencies (mm)")
        plt.close(fig)

# notebook outputs
df_noazi_all = pd.concat(all_noazi_tables, ignore_index=True)
print("NOAZI PCV tables created for", df_noazi_all["antenna_key"].nunique(), "antennas.")
display(df_noazi_all)


NOAZI PCV tables created for 2 antennas.


,antenna_key,freq,present,DAZI,zen1,zen2,dzen,n_z,NOAZI_present
0,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,C02,False,NaN,NaN,NaN,NaN,0,False
1,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,C06,False,NaN,NaN,NaN,NaN,0,False
2,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,E06,False,NaN,NaN,NaN,NaN,0,False
3,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,E07,False,NaN,NaN,NaN,NaN,0,False
4,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,E08,False,NaN,NaN,NaN,NaN,0,False
5,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G01,True,5.0,0.0,90.0,5.0,19,True
6,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G02,True,5.0,0.0,90.0,5.0,19,True
7,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G05,False,NaN,NaN,NaN,NaN,0,False
8,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R01,True,5.0,0.0,90.0,5.0,19,True
9,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R02,True,5.0,0.0,90.0,5.0,19,True


In [10]:
# =========================================
# CELL 9 — Plot: PCV 2D heatmaps (azimuth × zenith) as POLAR (azimuth × elevation)
#   - Outer ring: elevation 0° (zenith 90°)
#   - Center:     elevation 90° (zenith 0°)
#   - Uses pcolormesh on a polar axis for correct binning.
# =========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

if SELECTIONS["selected_mode"] != "2D":
    print("Selected mode is not 2D. Skipping Cell 9.")
else:
    selected_freqs = SELECTIONS["selected_freqs"]

    # --- user control to avoid too many plots ---
    print("Heatmap generation control:")
    print("  a) all available (per antenna, per selected frequency)")
    print("  b) limit total number of heatmaps (max N per antenna)")
    heat_choice = input("Enter a/b: ").strip().lower()

    max_per_ant = None
    if heat_choice == "b":
        max_per_ant = int(input("Enter max heatmaps per antenna (integer >=1): ").strip())
        if max_per_ant < 1:
            raise ValueError("max must be >= 1")

    def _edges_from_centers(x):
        """
        Create bin edges from bin centers (monotonic assumed).
        Works for pcolormesh to avoid distortion at bin boundaries.
        """
        x = np.asarray(x, dtype=float)
        if len(x) < 2:
            return np.array([x[0] - 0.5, x[0] + 0.5], dtype=float)
        dx = np.diff(x)
        edges = np.empty(len(x) + 1, dtype=float)
        edges[1:-1] = x[:-1] + dx / 2.0
        edges[0] = x[0] - dx[0] / 2.0
        edges[-1] = x[-1] + dx[-1] / 2.0
        return edges

    def build_heatmap_matrix(fdata: dict):
        """
        Returns (azi_sorted, z_sorted, M) where:
          - azi_sorted in degrees (centers)
          - z_sorted   in degrees (zenith centers)
          - M shape    = (n_azi, n_z), mm values
        """
        z = fdata.get("z")
        pcv_azi = fdata.get("pcv_azi")
        dazi = fdata.get("dazi")

        if z is None or pcv_azi is None or len(pcv_azi) == 0:
            return None, None, None
        if dazi is None or dazi <= 0:
            return None, None, None

        azis = sorted(pcv_azi.keys())
        M = np.vstack([pcv_azi[a] for a in azis])  # (n_azi, n_z)

        if M.shape[1] != len(z):
            return None, None, None

        # ensure sorted zenith too (should already be)
        z_arr = np.array(z, dtype=float)
        z_sort = np.argsort(z_arr)
        z_arr = z_arr[z_sort]
        M = M[:, z_sort]

        return np.array(azis, dtype=float), z_arr, M.astype(float)

    def plot_polar_heatmap(azi_deg, z_deg, M_mm, ant_key: str, freq: str):
        """
        Polar heatmap:
          theta = azimuth (deg)
          r     = zenith (deg)  -> outer=90 => elevation 0
        Radial tick labels are shown as elevation degrees (90 - zenith).
        """
        azi_deg = np.asarray(azi_deg, dtype=float)
        z_deg   = np.asarray(z_deg, dtype=float)
        M_mm    = np.asarray(M_mm, dtype=float)

        # sort azimuth ascending
        a_sort = np.argsort(azi_deg)
        azi_deg = azi_deg[a_sort]
        M_mm = M_mm[a_sort, :]

        # build edges
        a_edges = _edges_from_centers(azi_deg)
        z_edges = _edges_from_centers(z_deg)

        theta_edges = np.deg2rad(a_edges)  # radians for polar axis
        r_edges = z_edges                 # radius = zenith (deg)

        # mesh for pcolormesh: (n_azi+1, n_z+1)
        TH, RR = np.meshgrid(theta_edges, r_edges, indexing="ij")

        fig = plt.figure(figsize=(7.2, 7.2))
        ax = plt.subplot(111, projection="polar")

        # geomatics-friendly orientation: 0° at North, clockwise azimuth
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)

        pcm = ax.pcolormesh(TH, RR, M_mm, shading="auto")

        # radial: 0 (zenith) at center, 90 (horizon) at edge
        ax.set_ylim(0, 90)

        # label radial ticks as ELEVATION (deg) instead of zenith
        rticks = np.array([0, 15, 30, 45, 60, 75, 90], dtype=float)  # zenith ticks
        ax.set_yticks(rticks)
        ax.set_yticklabels([f"{int(90 - r)}°" for r in rticks])      # elevation labels
        ax.set_ylabel("Elevation (deg)", labelpad=20)

        ax.set_title(f"PCV polar heatmap (mm) — {ant_key} — {freq}\n(outer=elev 0°, center=elev 90°)", va="bottom")

        cb = plt.colorbar(pcm, pad=0.10)
        cb.set_label("PCV (mm)")

        plt.tight_layout()

        safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", ant_key)[:120]
        fname = f"pcv_polar_heatmap_{safe}_{freq}.png"
        return fig, fname

    # --- write to report ---
    report_append("<hr>")
    r_h2("PCV 2D polar heatmaps (azimuth × elevation)")

    plotted_rows = []

    for ant_key, ant in ant_data.items():
        r_h2(f"Polar heatmaps — {ant_key}")

        n_plotted = 0
        for f in selected_freqs:
            fdata = ant["freq"].get(f)
            if fdata is None:
                continue

            azi, z, M = build_heatmap_matrix(fdata)
            if M is None:
                continue

            fig, fname = plot_polar_heatmap(azi, z, M, ant_key, f)
            r_fig(fig, fname, caption=f"PCV polar heatmap for {f} (mm); grid: azimuth × zenith (displayed as elevation)")
            plt.close(fig)

            plotted_rows.append({
                "antenna_key": ant_key,
                "freq": f,
                "n_azi": int(len(azi)),
                "n_z": int(len(z)),
                "DAZI": float(fdata.get("dazi")) if fdata.get("dazi") is not None else np.nan
            })

            n_plotted += 1
            if (max_per_ant is not None) and (n_plotted >= max_per_ant):
                break

        if n_plotted == 0:
            r_p("No 2D PCV polar heatmaps available for the selected frequencies (missing azimuth grid).")

    df_heat = pd.DataFrame(plotted_rows)
    if not df_heat.empty:
        r_df(df_heat, caption="Polar heatmaps generated", index=False)

    print("Heatmaps generated:", 0 if df_heat.empty else len(df_heat))
    display(df_heat)


Heatmap generation control:
  a) all available (per antenna, per selected frequency)
  b) limit total number of heatmaps (max N per antenna)
Enter a/b: a
Heatmaps generated: 14


,antenna_key,freq,n_azi,n_z,DAZI
0,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G01,73,19,5.0
1,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,G02,73,19,5.0
2,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R01,73,19,5.0
3,LEIGS18 NONE__NONE__LEIGS18_NONE.atx,R02,73,19,5.0
4,LEIGS18 NONE__NONE__leigs18_________no...,C02,73,19,5.0
5,LEIGS18 NONE__NONE__leigs18_________no...,C06,73,19,5.0
6,LEIGS18 NONE__NONE__leigs18_________no...,E06,73,19,5.0
7,LEIGS18 NONE__NONE__leigs18_________no...,E07,73,19,5.0
8,LEIGS18 NONE__NONE__leigs18_________no...,E08,73,19,5.0
9,LEIGS18 NONE__NONE__leigs18_________no...,G01,73,19,5.0


In [11]:
# =========================================
# CELL 10 Optional comparison prompt + (if yes) run comparisons between two antennas:
#          ΔPCO (N/E/U), ΔPCV(NOAZI), and ΔPCV 2D heatmap (for common frequencies)
#          UPDATED: ΔPCV 2D heatmaps rendered as POLAR (azimuth × elevation; outer=elev0, center=elev90)
# =========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- require exactly two antennas for comparison ---
ant_keys = list(ant_data.keys())
if len(ant_keys) != 2:
    msg = f"Comparison requires exactly 2 antennas. Found {len(ant_keys)}. Skipping comparison."
    print(msg)
    report_append("<hr>")
    r_h2("Antenna comparison (skipped)")
    r_p(msg)
else:
    A_key, B_key = ant_keys[0], ant_keys[1]
    A, B = ant_data[A_key], ant_data[B_key]

    explanation = (
        "ANTEX comparison concept (PCO/PCV)\n"
        "A) It is generally NOT meaningful as a 'which antenna is better' comparison between different antenna models, because:\n"
        "   1) it does not answer 'which is better' (it is not a quality metric),\n"
        "   2) it does not automatically translate to 'error' or 'accuracy',\n"
        "   3) PCV/PCO are correction models, not performance measurements.\n"
        "B) It IS meaningful mainly for:\n"
        "   - comparing ANTEX calibrations from different providers for the same antenna model, and/or\n"
        "   - comparing different realizations/versions of the same model (e.g., with/without radome).\n"
    )

    print(explanation)
    # write explanation to report (before user choice)
    report_append("<hr>")
    r_h2("Antenna comparison prompt")
    r_pre(explanation)
    do_compare = input("Run antenna comparison (ΔPCO, ΔPCV NOAZI, ΔPCV 2D)? (y/n): ").strip().lower()
    r_p(f"User choice: {do_compare}")

    if do_compare != "y":
        print("Skipping comparison by user choice.")
        r_h2("Antenna comparison (skipped)")
        r_p("User chose not to run antenna comparison (ΔPCO/ΔPCV).")
    else:
        # Use common frequencies (intersection) and respect selected frequencies
        common = set(A["freq"].keys()).intersection(set(B["freq"].keys()))
        selected = set(SELECTIONS["selected_freqs"])
        common_selected = sorted(list(common.intersection(selected)))

        if len(common_selected) == 0:
            raise RuntimeError("No common frequencies between the two antennas within the selected frequencies.")

        print("Comparing antennas:")
        print("  A:", A_key)
        print("  B:", B_key)
        print("Common frequencies used:", ", ".join(common_selected))

        # -------------------------------------------------
        # ΔPCO table + bar plot
        # -------------------------------------------------
        def delta_pco_table(A, B, freqs):
            rows = []
            for f in freqs:
                pA = A["freq"][f].get("pco")
                pB = B["freq"][f].get("pco")
                if pA is None or pB is None:
                    rows.append({"freq": f, "dN_mm": np.nan, "dE_mm": np.nan, "dU_mm": np.nan, "present": False})
                else:
                    rows.append({
                        "freq": f,
                        "dN_mm": float(pA[0]) - float(pB[0]),
                        "dE_mm": float(pA[1]) - float(pB[1]),
                        "dU_mm": float(pA[2]) - float(pB[2]),
                        "present": True
                    })
            return pd.DataFrame(rows)

        def plot_delta_pco(df, title):
            dfp = df[df["present"]].copy()
            if dfp.empty:
                return None, None
            x = np.arange(len(dfp))
            w = 0.25
            fig = plt.figure(figsize=(12, 4.5))
            plt.bar(x - w, dfp["dN_mm"].values, width=w, label="ΔNorth (mm)")
            plt.bar(x,     dfp["dE_mm"].values, width=w, label="ΔEast (mm)")
            plt.bar(x + w, dfp["dU_mm"].values, width=w, label="ΔUp (mm)")
            plt.xticks(x, dfp["freq"].values)
            plt.xlabel("Frequency code")
            plt.ylabel("ΔPCO (mm)  [A - B]")
            plt.title(title)
            plt.legend()
            plt.grid(True, axis="y", linestyle="--", linewidth=0.5)
            plt.tight_layout()
            fname = "delta_pco.png"
            return fig, fname

        df_dpco = delta_pco_table(A, B, common_selected)

        # -------------------------------------------------
        # ΔPCV(NOAZI) plot (per frequency)
        # -------------------------------------------------
        def build_delta_noazi_series(A, B, freqs):
            series = {}
            meta = []
            for f in freqs:
                fA = A["freq"][f]
                fB = B["freq"][f]
                zA, pA = fA.get("z"), fA.get("pcv_noazi")
                zB, pB = fB.get("z"), fB.get("pcv_noazi")
                ok = (zA is not None and zB is not None and pA is not None and pB is not None and
                      len(zA) == len(zB) == len(pA) == len(pB) and np.allclose(zA, zB))
                meta.append({"freq": f, "NOAZI_A": pA is not None, "NOAZI_B": pB is not None, "ok": bool(ok)})
                if ok:
                    series[f] = (np.array(zA, dtype=float), np.array(pA, dtype=float) - np.array(pB, dtype=float))
            return pd.DataFrame(meta), series

        def plot_delta_noazi(series, title):
            if len(series) == 0:
                return None, None
            fig = plt.figure(figsize=(12, 5))
            for f, (z, dp) in series.items():
                plt.plot(z, dp, label=f)
            plt.xlabel("Zenith angle (deg)")
            plt.ylabel("ΔPCV (mm)  [A - B]")
            plt.title(title)
            plt.grid(True, linestyle="--", linewidth=0.5)
            plt.legend(ncol=3, fontsize=9)
            plt.tight_layout()
            fname = "delta_pcv_noazi.png"
            return fig, fname

        df_noazi_meta, dnoazi_series = build_delta_noazi_series(A, B, common_selected)

        # -------------------------------------------------
        # ΔPCV 2D heatmap utilities (POLAR)
        # -------------------------------------------------
        def heatmap_matrix(fdata: dict):
            z = fdata.get("z")
            pcv_azi = fdata.get("pcv_azi")
            dazi = fdata.get("dazi")
            if z is None or pcv_azi is None or len(pcv_azi) == 0:
                return None, None, None
            if dazi is None or dazi <= 0:
                return None, None, None

            azis = sorted(pcv_azi.keys())
            M = np.vstack([pcv_azi[a] for a in azis])  # (n_azi, n_z)
            if M.shape[1] != len(z):
                return None, None, None

            z_arr = np.array(z, dtype=float)
            z_sort = np.argsort(z_arr)
            z_arr = z_arr[z_sort]
            M = M[:, z_sort]

            return np.array(azis, dtype=float), z_arr, M.astype(float)

        def _edges_from_centers(x):
            x = np.asarray(x, dtype=float)
            if len(x) < 2:
                return np.array([x[0] - 0.5, x[0] + 0.5], dtype=float)
            dx = np.diff(x)
            edges = np.empty(len(x) + 1, dtype=float)
            edges[1:-1] = x[:-1] + dx / 2.0
            edges[0] = x[0] - dx[0] / 2.0
            edges[-1] = x[-1] + dx[-1] / 2.0
            return edges

        def plot_delta_heatmap_polar(azi_deg, z_deg, dM, freq, title_prefix):
            """
            Polar:
              theta = azimuth (deg)
              r     = zenith (deg) => outer=90 (elev 0), center=0 (elev 90)
            """
            azi_deg = np.asarray(azi_deg, dtype=float)
            z_deg   = np.asarray(z_deg, dtype=float)
            dM      = np.asarray(dM, dtype=float)

            a_sort = np.argsort(azi_deg)
            azi_deg = azi_deg[a_sort]
            dM = dM[a_sort, :]

            a_edges = _edges_from_centers(azi_deg)
            z_edges = _edges_from_centers(z_deg)

            theta_edges = np.deg2rad(a_edges)
            r_edges = z_edges  # zenith radius

            TH, RR = np.meshgrid(theta_edges, r_edges, indexing="ij")

            fig = plt.figure(figsize=(7.2, 7.2))
            ax = plt.subplot(111, projection="polar")

            ax.set_theta_zero_location("N")
            ax.set_theta_direction(-1)

            pcm = ax.pcolormesh(TH, RR, dM, shading="auto")
            ax.set_ylim(0, 90)

            rticks = np.array([0, 15, 30, 45, 60, 75, 90], dtype=float)  # zenith
            ax.set_yticks(rticks)
            ax.set_yticklabels([f"{int(90 - r)}°" for r in rticks])      # elevation labels
            ax.set_ylabel("Elevation (deg)", labelpad=20)

            ax.set_title(f"{title_prefix} — {freq}\n(outer=elev 0°, center=elev 90°)", va="bottom")

            cb = plt.colorbar(pcm, pad=0.10)
            cb.set_label("ΔPCV (mm)  [A - B]")

            plt.tight_layout()
            fname = f"delta_pcv_heatmap_{freq}.png"  # keep original filename pattern
            return fig, fname

        do_2d = (SELECTIONS["selected_mode"] == "2D")

        # --- write to report ---
        report_append("<hr>")
        r_h2("Antenna comparison (A - B)")
        r_p(f"A: {A_key}")
        r_p(f"B: {B_key}")
        r_p(f"Common frequencies used: {', '.join(common_selected)}")

        r_h2("ΔPCO (N/E/U) [A - B]")
        r_df(df_dpco, caption="ΔPCO per common frequency (mm); A - B", index=False)
        fig, fname = plot_delta_pco(df_dpco, f"ΔPCO (N/E/U) [A - B] — {A_key} minus {B_key}")
        if fig is not None:
            r_fig(fig, fname, caption="ΔPCO per frequency (mm)")
            plt.close(fig)

        r_h2("ΔPCV (NOAZI) vs Zenith [A - B]")
        r_df(df_noazi_meta, caption="NOAZI availability and grid compatibility check", index=False)
        fig, fname = plot_delta_noazi(dnoazi_series, f"ΔPCV (NOAZI) vs Zenith [A - B] — {A_key} minus {B_key}")
        if fig is not None:
            r_fig(fig, fname, caption="ΔPCV(NOAZI) curves per frequency (mm)")
            plt.close(fig)
        else:
            r_p("No compatible NOAZI series for ΔPCV plot.")

        r_h2("ΔPCV 2D heatmaps (polar: azimuth × elevation) [A - B]")
        if not do_2d:
            r_p("Selected mode is NOAZI; skipping 2D heatmap differences.")
        else:
            plotted = []
            for f in common_selected:
                aziA, zA, MA = heatmap_matrix(A["freq"][f])
                aziB, zB, MB = heatmap_matrix(B["freq"][f])

                ok = (
                    MA is not None and MB is not None and
                    len(aziA) == len(aziB) and len(zA) == len(zB) and
                    np.allclose(aziA, aziB) and np.allclose(zA, zB)
                )
                if not ok:
                    continue

                dM = MA - MB
                fig, fname = plot_delta_heatmap_polar(aziA, zA, dM, f, f"ΔPCV heatmap — {A_key} minus {B_key}")
                r_fig(fig, fname, caption=f"ΔPCV polar heatmap for {f} (mm)  [A - B]")
                plt.close(fig)
                plotted.append({"freq": f, "n_azi": int(len(aziA)), "n_z": int(len(zA))})

            df_dheat = pd.DataFrame(plotted)
            if df_dheat.empty:
                r_p("No compatible 2D heatmaps found for differences.")
            else:
                r_df(df_dheat, caption="ΔPCV 2D heatmaps generated (polar)", index=False)

        # notebook outputs
        print("\nΔPCO table:")
        display(df_dpco)

        print("\nΔPCV NOAZI availability:")
        display(df_noazi_meta)

        if do_2d:
            print("\nΔPCV 2D polar heatmaps were written to the report for compatible frequencies.")


ANTEX comparison concept (PCO/PCV)
A) It is generally NOT meaningful as a 'which antenna is better' comparison between different antenna models, because:
   1) it does not answer 'which is better' (it is not a quality metric),
   2) it does not automatically translate to 'error' or 'accuracy',
   3) PCV/PCO are correction models, not performance measurements.
B) It IS meaningful mainly for:
   - comparing ANTEX calibrations from different providers for the same antenna model, and/or
   - comparing different realizations/versions of the same model (e.g., with/without radome).

Run antenna comparison (ΔPCO, ΔPCV NOAZI, ΔPCV 2D)? (y/n): y
Comparing antennas:
  A: LEIGS18         NONE__NONE__LEIGS18_NONE.atx
  B: LEIGS18         NONE__NONE__leigs18_________none.atx
Common frequencies used: G01, G02, R01, R02

ΔPCO table:


,freq,dN_mm,dE_mm,dU_mm,present
0,G01,0.42,0.00,0.08,True
1,G02,0.04,0.22,1.62,True
2,R01,0.01,-0.07,-0.22,True
3,R02,0.33,0.23,1.54,True



ΔPCV NOAZI availability:


,freq,NOAZI_A,NOAZI_B,ok
0,G01,True,True,True
1,G02,True,True,True
2,R01,True,True,True
3,R02,True,True,True



ΔPCV 2D polar heatmaps were written to the report for compatible frequencies.


In [12]:
# =========================================
# CELL 11 — Finalize report + zip for download
# =========================================

from pathlib import Path
import shutil
from google.colab import files

# Close HTML
report_close()

# Paths (consistent with earlier cells)
REPORT_DIR = Path("/content/report")
REPORT_HTML = REPORT_DIR / "report.html"
ZIP_PATH = Path("/content/report.zip")

# Basic checks
if not REPORT_DIR.exists():
    raise FileNotFoundError(f"Report dir not found: {REPORT_DIR}")
if not REPORT_HTML.exists():
    raise FileNotFoundError(f"Report HTML not found: {REPORT_HTML}")

# (Re)create zip
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# shutil.make_archive returns the created archive path (without .zip in argument)
archive_base = str(ZIP_PATH).replace(".zip", "")
zip_created = shutil.make_archive(base_name=archive_base, format="zip", root_dir=str(REPORT_DIR))

print(f"Report finalized: {REPORT_HTML}")
print(f"ZIP created: {zip_created}")

# Trigger download in Colab
files.download(str(ZIP_PATH))


Report finalized: /content/report/report.html
ZIP created: /content/report.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**END OF PROGRAM**